# Track 2: Atlas-Free Coordinate GNN — A100 Training

**Hardware**: A100 40GB — Runtime → Change runtime type → A100 GPU  
**Output**: `best_coord_gnn.pt` + `last_coord_gnn.pt` → Google Drive

### Before you start
1. **Runtime → Change runtime type → A100 GPU**
2. Run all cells top-to-bottom.

### Resuming after a session crash
Re-run all cells from the top. The graph cache on Drive is reused automatically — only missing graphs are rebuilt.

## 0 — Runtime check

In [ ]:
import subprocess
gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu_info.returncode != 0:
    raise RuntimeError('No GPU — Runtime → Change runtime type → A100 GPU')
print(gpu_info.stdout)

import torch
assert torch.cuda.is_available()
print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'RAM : check via !cat /proc/meminfo | grep MemTotal')

## 1 — Install dependencies

In [ ]:
import torch, sys, subprocess
cuda_tag  = 'cu' + torch.version.cuda.replace('.', '')
torch_tag = torch.__version__.split('+')[0]
print(f'torch={torch_tag}  cuda={cuda_tag}')

In [ ]:
# Minimal installs — coordinates are a parquet, text embeddings are pre-computed .pt files.
# No nimare / transformers / adapters needed.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pyarrow', 'pandas', 'matplotlib', 'tqdm', 'umap-learn'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'nibabel', 'nilearn'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch_geometric'], check=True)

pyg_url = f'https://data.pyg.org/whl/torch-{torch_tag}+{cuda_tag}.html'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch_scatter', 'torch_sparse', 'torch_cluster', '-f', pyg_url], check=True)

print('Done.')

## 2 — Clone repo + patch __init__

In [ ]:
import os, sys
from pathlib import Path

REPO_DIR = Path('/content/neurovlm')
if not REPO_DIR.exists():
    !git clone --branch neurovlm_gnn https://github.com/neurovlm/neurovlm.git {REPO_DIR}
else:
    print('Repo exists — pulling.')
    !git -C {REPO_DIR} pull

src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Patch neurovlm/__init__.py so heavy optional deps aren't imported on load
(REPO_DIR / 'src/neurovlm/__init__.py').write_text(
    'from __future__ import annotations\n\n\n'
    'def __getattr__(name: str):\n'
    '    if name == "NeuroVLM":\n'
    '        from .core import NeuroVLM\n'
    '        return NeuroVLM\n'
    '    raise AttributeError(f"module {__name__!r} has no attribute {name!r}")\n'
)
print('sys.path:', src_dir)

## 3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR      = Path('/content/drive/MyDrive/neurovlm_track2')
CHECKPOINT_DIR = DRIVE_DIR / 'checkpoints/coord_gnn'
DRIVE_CACHE    = DRIVE_DIR / 'data/coord_graphs'    # persistent cache on Drive
LOCAL_CACHE    = Path('/content/coord_graphs')       # fast local SSD copy

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)

print(f'Drive cache  → {DRIVE_CACHE}')
print(f'Local cache  → {LOCAL_CACHE}  (fast reads during training)')
print(f'Checkpoints  → {CHECKPOINT_DIR}')

## 4 — Hyperparameters (A100-tuned)

In [ ]:
CFG = dict(
    # Graph construction
    K            = 7,
    MAX_DIST_MM  = 30.0,
    # Model — larger than local defaults to fill the GPU
    HIDDEN       = 256,    # was 128; doubles params from ~2.6M to ~10M
    HEADS        = 8,
    DROPOUT      = 0.1,
    # Training
    N_EPOCHS     = 200,
    BATCH_SIZE   = 512,    # 512 graphs × ~20 avg nodes = ~10K nodes/batch; fine for 40GB
    LR_GNN       = 1e-4,
    LR_PROJ      = 1e-5,
    WARMUP_EPOCHS= 15,
    TEMPERATURE  = 0.07,
    VAL_INTERVAL = 5,
    # DataLoader
    NUM_WORKERS  = 4,      # parallel graph loading from local SSD
    PIN_MEMORY   = True,   # page-locked memory for fast CPU→GPU transfer
    # BF16 AMP — A100 Tensor Cores run BF16 at ~2x FP32 speed
    USE_AMP      = True,
    SEED         = 42,
)
print('Config:', CFG)

## 5 — Load coordinates

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from neurovlm.retrieval_resources import _load_pubmed_coordinates
from neurovlm.gnn.coord_graph import normalize_coords, coords_to_graph

print('Loading peak coordinates …')
coords_df = _load_pubmed_coordinates()
print(f'  {coords_df.shape}  columns: {list(coords_df.columns)}')

peak_counts = coords_df.groupby('pmid').size()
print(f'\nPeak counts across {len(peak_counts):,} papers:')
print(f'  min={peak_counts.min()}  max={peak_counts.max()}')
print(f'  mean={peak_counts.mean():.1f}  median={peak_counts.median():.0f}')
print(f'  p5={peak_counts.quantile(0.05):.0f}  p95={peak_counts.quantile(0.95):.0f}')
print(f'  <3 peaks: {(peak_counts<3).sum()} ({100*(peak_counts<3).mean():.1f}%)')

## 6 — Build Drive cache, then copy to local SSD

**Why two copies?**  
Drive has ~100ms latency per small file read. Local Colab SSD has <1ms.  
Reading 30K `.pt` files from Drive during every epoch would add hours per epoch.  
We build once on Drive (persistent across sessions), then mirror to local SSD (fast reads).

In [ ]:
from neurovlm.data import load_dataset, load_latent

print('Loading SPECTER text embeddings …')
text_latents = load_latent('pubmed_text')
pubmed_df    = load_dataset('pubmed_text')

if isinstance(text_latents, tuple):
    text_tensor, pmids_arr = text_latents
    unique_pmids = np.asarray(pmids_arr).astype(str)
elif isinstance(text_latents, dict):
    pmid_list    = list(text_latents.keys())
    text_tensor  = torch.stack([torch.tensor(text_latents[p], dtype=torch.float32)
                                for p in pmid_list])
    unique_pmids = np.array([str(p) for p in pmid_list])
else:
    text_tensor  = text_latents if isinstance(text_latents, torch.Tensor) \
                   else torch.tensor(text_latents)
    unique_pmids = pubmed_df['pmid'].astype(str).values[:len(text_tensor)] \
                   if 'pmid' in pubmed_df.columns \
                   else np.arange(len(text_tensor)).astype(str)

print(f'  text_tensor  : {text_tensor.shape}')
print(f'  unique_pmids : {len(unique_pmids):,}')

In [ ]:
# Step 1: Build (or verify) the cache on Google Drive
from neurovlm.gnn.coord_dataset import CoordGraphDataset

print('Step 1: Building graph cache on Drive (reuses existing files) …')
ds_drive = CoordGraphDataset(
    coords_df       = coords_df,
    text_embeddings = text_tensor,
    unique_pmids    = unique_pmids,
    cache_dir       = str(DRIVE_CACHE),
    k               = CFG['K'],
    max_dist_mm     = CFG['MAX_DIST_MM'],
    preload_to_ram  = False,   # don't preload yet — Drive reads are slow
)
print(f'Drive cache complete: {len(ds_drive):,} papers')

In [ ]:
# Step 2: Mirror Drive cache → local SSD  (rsync-style: only copies missing files)
import shutil
drive_files  = set(DRIVE_CACHE.glob('paper_*.pt'))
local_files  = set(LOCAL_CACHE.glob('paper_*.pt'))
local_names  = {f.name for f in local_files}
to_copy      = [f for f in drive_files if f.name not in local_names]

print(f'Local SSD: {len(local_files)} cached, {len(to_copy)} to copy from Drive …')
for src in to_copy:
    shutil.copy2(src, LOCAL_CACHE / src.name)
print(f'SSD cache ready: {len(list(LOCAL_CACHE.glob("paper_*.pt"))):,} files')

In [ ]:
# Step 3: Create the training dataset from LOCAL SSD + preload everything to RAM
# 30K graphs × ~2KB each ≈ 60MB — trivial on A100 (83GB system RAM)
print('Step 3: Loading dataset from local SSD + preloading all graphs to RAM …')
ds = CoordGraphDataset(
    coords_df       = coords_df,
    text_embeddings = text_tensor,
    unique_pmids    = unique_pmids,
    cache_dir       = str(LOCAL_CACHE),
    k               = CFG['K'],
    max_dist_mm     = CFG['MAX_DIST_MM'],
    preload_to_ram  = True,   # eliminates ALL file I/O during training
)
print(f'Dataset ready: {len(ds):,} papers')

In [ ]:
# Verify PyG batching + shape check
from torch_geometric.loader import DataLoader
probe = next(iter(DataLoader(ds, batch_size=4, shuffle=False)))
print(f'x.shape          = {probe.x.shape}   (total_nodes, 5)')
print(f'edge_attr.shape  = {probe.edge_attr.shape}  (E, 4)')
print(f'y.shape          = {probe.y.shape}')
assert probe.x.shape[1] == 5 and probe.edge_attr.shape[1] == 4
print('✓ shapes correct')

torch.manual_seed(CFG['SEED'])
train_ds, val_ds, test_ds = ds.split(val_frac=0.1, test_frac=0.1, seed=CFG['SEED'])
print(f'Split: train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_ds):,}')

## 7 — Model

In [ ]:
from torch_geometric.data import Data, Batch
from neurovlm.gnn.coord_model import CoordGNN
from neurovlm.gnn.model import TextProjHead

brain_encoder = CoordGNN(
    in_dim=5, hidden=CFG['HIDDEN'], heads=CFG['HEADS'],
    out_dim=384, dropout=CFG['DROPOUT'],
)
text_proj = TextProjHead(in_dim=768, hidden_dim=512, out_dim=384)

n_brain = brain_encoder.count_parameters()
n_text  = sum(p.numel() for p in text_proj.parameters())
print(f'CoordGNN params     : {n_brain:,}')
print(f'TextProjHead params : {n_text:,}')

# Shape check
dummy = Data(x=torch.randn(10,5),
             edge_index=torch.tensor([[0,1,2],[1,2,0]], dtype=torch.long),
             edge_attr=torch.randn(3,4))
b = Batch.from_data_list([dummy, dummy])
out = brain_encoder(b.x, b.edge_index, b.edge_attr, b.batch)
assert out.shape == (2, 384)
print(f'✓ Forward pass: (20,5) → {tuple(out.shape)}')

In [ ]:
# Optional: torch.compile for ~1.5x extra speedup on A100
# May print warnings about graph breaks with PyG — safe to ignore.
try:
    brain_encoder = torch.compile(brain_encoder, mode='reduce-overhead')
    text_proj     = torch.compile(text_proj,     mode='reduce-overhead')
    print('torch.compile enabled (reduce-overhead mode)')
except Exception as e:
    print(f'torch.compile skipped: {e}')

## 8 — Training

**A100 optimizations active:**
- `batch_size=512` (was 128 locally) — fills GPU compute
- `preload_to_ram=True` — zero file I/O per batch, graphs live in system RAM
- `num_workers=4` — parallel CPU prefetch while GPU trains
- `pin_memory=True` — fast page-locked CPU→GPU transfer
- `use_amp=True` — BF16 on A100 Tensor Cores (~2x vs FP32, no overflow risk)

In [ ]:
# VRAM baseline before training
torch.cuda.reset_peak_memory_stats()
print(f'VRAM before training: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated')

In [ ]:
from neurovlm.gnn.coord_train import CoordTrainer

trainer = CoordTrainer(
    brain_encoder  = brain_encoder,
    text_proj      = text_proj,
    lr_gnn         = CFG['LR_GNN'],
    lr_proj        = CFG['LR_PROJ'],
    batch_size     = CFG['BATCH_SIZE'],
    n_epochs       = CFG['N_EPOCHS'],
    warmup_epochs  = CFG['WARMUP_EPOCHS'],
    temperature    = CFG['TEMPERATURE'],
    device         = 'cuda',
    val_interval   = CFG['VAL_INTERVAL'],
    checkpoint_dir = str(CHECKPOINT_DIR),
    num_workers    = CFG['NUM_WORKERS'],
    pin_memory     = CFG['PIN_MEMORY'],
    use_amp        = CFG['USE_AMP'],
    verbose        = True,
)

trainer.fit(train_ds, val_ds)

In [ ]:
# VRAM report after training
peak_gb  = torch.cuda.max_memory_allocated() / 1e9
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'Peak VRAM used: {peak_gb:.2f} / {total_gb:.1f} GB  '
      f'({100*peak_gb/total_gb:.1f}% utilization)')
if peak_gb / total_gb < 0.5:
    print('Tip: VRAM <50% used — try BATCH_SIZE=1024 or HIDDEN=384 next run.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(trainer.history['train_loss'])
axes[0].set_title('Train Loss (InfoNCE)'); axes[0].set_xlabel('Epoch')

axes[1].plot(trainer.history['val_recall_t2i'], label='t2i')
axes[1].plot(trainer.history['val_recall_i2t'], label='i2t')
axes[1].set_title('Val Recall AUC')
axes[1].set_xlabel(f'Val step (every {CFG["VAL_INTERVAL"]} epochs)'); axes[1].legend()

axes[2].plot(trainer.history['embed_sim'], color='red')
axes[2].axhline(0.95, color='black', ls='--', label='collapse threshold')
axes[2].set_title('embed_sim (collapse monitor)'); axes[2].legend()

plt.tight_layout()
plt.savefig(str(DRIVE_DIR / 'training_curve.png'), dpi=150)
plt.show()

## 9 — Evaluation

In [ ]:
from neurovlm.metrics import recall_at_k, recall_curve

trainer.restore_best()
results = {}

for split_name, split_ds in [('Val', val_ds), ('Test', test_ds)]:
    brain_emb, text_emb = trainer.collect_embeddings(split_ds)
    sim = F.normalize(text_emb, dim=1) @ F.normalize(brain_emb, dim=1).T

    r1  = recall_at_k(sim, 1)
    r5  = recall_at_k(sim, 5)
    r10 = recall_at_k(sim, 10)
    t2i, i2t = recall_curve(text_emb, brain_emb)
    auc = float((t2i.mean() + i2t.mean()) / 2)

    results[split_name] = dict(r1=r1, r5=r5, r10=r10, auc=auc)
    print(f'\n{split_name} ({len(split_ds)} papers):')
    print(f'  recall@1  = {r1:.4f}')
    print(f'  recall@5  = {r5:.4f}')
    print(f'  recall@10 = {r10:.4f}')
    print(f'  AUC       = {auc:.4f}  (NeuroVLM MLP baseline ≈ 0.81)')

print('\n┌───────────────────────────┬──────────┬────────────┐')
print('│ Model                     │ Test AUC │ Atlas-free │')
print('├───────────────────────────┼──────────┼────────────┤')
print('│ NeuroVLM MLP (baseline)   │ 0.81     │ No         │')
print('│ Track 1 DiFuMo GAT        │ (run it) │ No         │')
print(f'│ Track 2 Coord GNN (ours)  │ {results["Test"]["auc"]:.4f}   │ Yes        │')
print('└───────────────────────────┴──────────┴────────────┘')

## 10 — Attention analysis

In [ ]:
snapshots = trainer.get_attention_snapshot(test_ds, n_samples=10)
for snap in snapshots:
    n_peaks = snap['node_coords_mni'].shape[0]
    print(f'\nPaper {snap["paper_idx"]} ({n_peaks} peaks):')
    for e in snap['top_edges']:
        dist_mm = float(np.linalg.norm(np.array(e['src_mni']) - np.array(e['dst_mni'])))
        s = [f'{v:.0f}' for v in e['src_mni']]
        d = [f'{v:.0f}' for v in e['dst_mni']]
        print(f'  [{" ".join(s)}] → [{" ".join(d)}]  attn={e["weight"]:.4f}  dist={dist_mm:.0f}mm')

## 11 — Failure analysis

In [ ]:
brain_emb, text_emb = trainer.collect_embeddings(test_ds)
sim = F.normalize(text_emb, dim=1) @ F.normalize(brain_emb, dim=1).T
top10_hits = (sim.argsort(dim=1, descending=True)[:, :10] ==
              torch.arange(len(sim)).unsqueeze(1)).any(dim=1)

worst_pos = (~top10_hits).nonzero(as_tuple=True)[0][:20].tolist()
print(f'recall@10=0: {(~top10_hits).sum()} / {len(sim)} test papers')

worst_loader = DataLoader([test_ds[i] for i in worst_pos], batch_size=1, shuffle=False)
peak_counts_worst = [g.x.shape[0] for g in worst_loader]
print(f'Worst-paper peak counts: mean={np.mean(peak_counts_worst):.1f}  '
      f'min={min(peak_counts_worst)}  max={max(peak_counts_worst)}')

## 12 — UMAP

In [ ]:
try:
    import umap
    brain_emb, _ = trainer.collect_embeddings(test_ds)
    emb_np = F.normalize(brain_emb, dim=1).cpu().numpy()
    N = min(500, len(emb_np))
    idx = np.random.default_rng(42).choice(len(emb_np), N, replace=False)
    emb_2d = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(emb_np[idx])
    fig, ax = plt.subplots(figsize=(8,6))
    ax.scatter(emb_2d[:,0], emb_2d[:,1], s=5, alpha=0.6, c='steelblue')
    ax.set_title(f'UMAP — CoordGNN ({N} test papers)')
    plt.tight_layout()
    plt.savefig(str(DRIVE_DIR / 'umap_coord_gnn.png'), dpi=150)
    plt.show()
except ImportError:
    print('umap-learn not available.')

## 13 — Save artifacts

In [ ]:
import json

with open(DRIVE_DIR / 'training_history.json', 'w') as f:
    json.dump({k: [float(v) for v in vals]
               for k, vals in trainer.history.items()}, f, indent=2)

with open(DRIVE_DIR / 'eval_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Saved to Drive:')
for p in sorted(DRIVE_DIR.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(DRIVE_DIR)}  ({p.stat().st_size/1e6:.1f} MB)')